# Practice 4 — MNIST, and the effect of model size

*Exposome Analytics Summer School, London 2026 — Deep Learning day.*

> **Before you run anything.** This notebook trains **17 different models** in
> order to compare their sizes. Switch to a GPU first:
> **Runtime → Change runtime type → T4 GPU**. The number of epochs has been
> lowered from 25 to 10 for the session; raise it again at home if you want the
> curves to settle.

If you are short of time, run the notebook down to *Prediction* and read the
rest: the model-complexity comparison is worth seeing even without running it.


# Mathematical Engineering of Deep Learning
----

## Practice 4 on MNIST DATA (Python version)


# Goals
----
In this tutorial, we mainly use the MNIST dataset to explore classification deep neural networks (DNN) models.
At the end of this tutorial, you should be comfortable to use a software package (here **keras**) to run different models for a classification task.  You will explore different models by exploring/tuning different hyperparamaters of the DNN:

- number of layers and nodes
- batch normalization
- regularization technique
- dropout
- weight initialization
- Early stopping




# MNIST Data set
----
## Loading package and dataset

First you have to install **keras** R packages using

In [ ]:
from keras.datasets import mnist
(train_X,train_Y), (test_X,test_Y) = mnist.load_data()
print('Training data shape : ', train_X.shape, train_Y.shape)
print('Testing data shape : ', test_X.shape, test_Y.shape)

In [ ]:
# Find the unique numbers from the train labels
import numpy as np

classes = np.unique(train_Y)
nClasses = len(classes)
print('Total number of classes : ', nClasses)
print('Classes : ', classes)


In [ ]:
# Display the first image in training data
import matplotlib.pyplot as plt

plt.figure(figsize=[5,5])
plt.subplot(121)
plt.imshow(train_X[0,:,:], cmap='gray')
plt.title("Class : {}".format(train_Y[0]))

In [ ]:
train_X[0]

**Reminder:** Keep in mind that all features need to be numeric for running a feedforward DNN. When you have some categorical features you have to transform into numerical values such as one-hot encoded.

## Scale the data set

The data is in gray scale with each image having a value between 0 and 255. We therefore need to normalize the data by 255.

input for reshape: 60000, 28, 28
output of reshape: 60000, 784

In [ ]:
a = np.array([[[1,2,3],[4,5,6], [5,6,7]], [[1.1,2,3.3],[4,5,6], [5,6,7]]])
a.shape
b = a.reshape(2, 9)
b.shape
print(a)

In [ ]:
print(b)

In [ ]:
##### Changing the type
train_X = train_X.astype('float32')
test_X = test_X.astype('float32')

##### Data processing
train_X = train_X.reshape(train_X.shape[0], -1)
test_X = test_X.reshape(test_X.shape[0], -1)

#### Normalizing the data
train_X = train_X / 255
test_X = test_X / 255

In [ ]:
print(train_X.shape, test_X.shape)

In [ ]:
train_X[0]


##  Transform the label data

For multi-classification model (multinomial response 0 to 9), Keras uses one-hot encoded for the outcome. For example, the digit 5 image that we have plotted above has a label of 5, so for all the digit 5 images, the one hot encoding vector would be $[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]$.

In [ ]:
# Change the labels from categorical to one-hot encoding
from tensorflow.keras.utils import to_categorical

train_Y_one_hot = to_categorical(train_Y)
test_Y_one_hot = to_categorical(test_Y)


In [ ]:
# Cross check
print('First image\'s class:', train_Y[0])
print('First image\'s one hot encoding:', train_Y_one_hot[0])


#  Implementation of a DNN using Keras
----

## Procedure
  * Initiate a sequential feed-forward DNN using keras.model.sequential()
  * Add some dense layers.


In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input

model = Sequential()
model.add(Input(shape=(train_X[0].size,)))
model.add(Dense(128, activation="relu"))
model.add(Dense(64, activation="relu"))
model.add(Dense(10, activation="softmax"))

Here, we have two hidden layers:

  * 128 neurons for the first layer
  * 64 for the second
  * 10 neurons for the output layer

Note that the **shape** in **Input** argument represents the number of features in the data (here 784). It is natural to choose the **softmax** function for the output layer. It is the most common to use **ReLU** activation for hidden layer.

## Backpropagation and Optimizer

We have to define our objective function to optimize and the optimizer to get a solution. The natural choise here is the cross entropy for categorical outcome. In lecture 3 you have learned different variants of the gradient descent. Keras offers several optimizers:

  * Stochastic gradient descent (sgd) optimizer
  * Adaptive Moment Estimation (adam)
  * RMSprop
  * Adaptive learning rate (Adadelta)


In [ ]:
# Compile
import keras
model.compile(loss=keras.losses.categorical_crossentropy, optimizer=keras.optimizers.RMSprop(), metrics=['accuracy'])
model.summary()

## Train our model

We will train our model on 25 epochs and a bath size of 128. We also use 20%
of our data for the validation step during the training phase, meaning that 60,000×0.2=12,000 of the samples are using for the validation step while 48,000 samples are used for the optimization step.

**Reminder:** An epoch describes the number of times the algorithm sees the entire data set. So with a batch size of 128, one epoch is achieved after 48,000/128 = 375 passes.

In [ ]:
# Train-validation Split of the training data
from sklearn.model_selection import train_test_split

np.random.seed(1)
train_train_X, train_valid_X, train_train_label, train_valid_label = train_test_split(train_X, train_Y_one_hot, test_size=0.2, random_state=42, shuffle=True)

print(train_train_X.shape, train_valid_X.shape, train_train_label.shape, train_valid_label.shape)

In [ ]:
fit1 = model.fit(train_train_X, train_train_label, batch_size=128, epochs=10, verbose=1, validation_data=(train_valid_X, train_valid_label))

In [ ]:
test_eval = model.evaluate(test_X, test_Y_one_hot, verbose=0)


print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

accuracy = fit1.history['accuracy']
val_accuracy = fit1.history['val_accuracy']
loss = fit1.history['loss']
val_loss = fit1.history['val_loss']
epochs = range(len(accuracy))
plt.plot(epochs, accuracy, '-b', label='Training accuracy')
plt.plot(epochs, val_accuracy, '-g', label='Validation accuracy')
plt.title('Training and validation accuracies')
plt.legend()
plt.figure()
plt.plot(epochs, loss, '-b', label='Training loss')
plt.plot(epochs, val_loss, '-g', label='Validation loss')
#plt.plot([6.0, 6.0], [0.0, val_loss[6]], '-r')
#plt.plot([6.0], [val_loss[6]], 'or')
plt.text(4.0, 0.11, "Early Stopping Point", fontsize=20)
plt.xlim(0, 25)
plt.ylim(0.0, 0.37)
#plt.title('Training and validation losses', fontsize=20)
plt.rcParams["figure.figsize"] = (12,7)
plt.legend(fontsize=20)
plt.show()

We can see that the loss function improves rapidly. However, we can see a potential overfit after 10 epochs. Indeed, the accurary rate of the validation set presents a flat shape after 10 epochs.

## Prediction

We can predict now the class (digits) for a new image.

In [ ]:
predicted_classes = model.predict(test_X[0:20])
predicted_classes

In [ ]:
sum(predicted_classes[2])

Observe that the predictions are floating point values, it is difficult to compare them with true test labels. So we now round off the output into integers.

In [ ]:
predicted_classes = np.argmax(np.round(predicted_classes),axis=1)
predicted_classes

In [ ]:
# Comparing with true labels
test_Y[0:20]

In [ ]:
# For the entire test data
predicted_classes = model.predict(test_X)
predicted_classes = np.argmax(np.round(predicted_classes),axis=1)


In [ ]:
np.unique(predicted_classes, return_counts=True)

In [ ]:
np.unique(test_Y, return_counts=True)

In [ ]:
import sklearn.metrics
cm = sklearn.metrics.confusion_matrix(test_Y, predicted_classes)

In [ ]:
print(cm)

In [ ]:
print('Accuracy:', sum(np.diag(cm))/test_X.shape[0])

# Improve our model by tuning some parameters
-----

## Model complexity

We will explore different model size by playing with the number of hidden layers from 1 to 3 and different number of neurons. Complex models have higher capacity to learn more features and patterns in the data, however they can overfit the training data. We try to maximize a high validation performance while minimizing the complexity of our model. The folowing table presents the 9 models that we will explore:

In [ ]:
from google.colab import data_table
import pandas as pd


df = pd.DataFrame({'Size':['1 Hidden Layer','2 Hidden Layers','3 Hidden Layers'],
                   'Small':[16,(16, 8), (16, 8, 4)],
                   'Medium':[64, (64, 32), (64, 32, 16)],
                   'Large': [256 , (256, 128), (256, 128, 64)]})

data_table.DataTable(df, include_index=False)

We have 9 models to run !!! better to wrap it into a nice function.


In [ ]:
def compiler(mdl):
  mdl.compile(loss=keras.losses.categorical_crossentropy, optimizer=keras.optimizers.RMSprop(), metrics=['accuracy'])
  mdl.summary()
  return

def trainer(mdl):
  ft = mdl.fit(train_train_X, train_train_label, batch_size=128, epochs=10, verbose=1, validation_data=(train_valid_X, train_valid_label))
  return ft

## One layer model

In [ ]:
## Small model with 1 layer
small_1_layer_model = Sequential()
small_1_layer_model.add(Input(shape=(train_X[0].size,)))
small_1_layer_model.add(Dense(16, activation="relu"))
small_1_layer_model.add(Dense(10, activation="softmax"))
compiler(small_1_layer_model)
fit_s_1 = trainer(small_1_layer_model)

## Medium model with 1 layer
medium_1_layer_model = Sequential()
medium_1_layer_model.add(Input(shape=(train_X[0].size,)))
medium_1_layer_model.add(Dense(64, activation="relu"))
medium_1_layer_model.add(Dense(10, activation="softmax"))
compiler(medium_1_layer_model)
fit_m_1 = trainer(medium_1_layer_model)

## Large model with 1 layer
large_1_layer_model = Sequential()
large_1_layer_model.add(Input(shape=(train_X[0].size,)))
large_1_layer_model.add(Dense(256, activation="relu"))
large_1_layer_model.add(Dense(10, activation="softmax"))
compiler(large_1_layer_model)
fit_l_1 = trainer(large_1_layer_model)

In [ ]:
## function for plotting the results

def plot_results(mdl, ft):
  test_eval = mdl.evaluate(test_X, test_Y_one_hot, verbose=0)
  print('Test loss:', test_eval[0])
  print('Test accuracy:', test_eval[1])

  accuracy = ft.history['accuracy']
  val_accuracy = ft.history['val_accuracy']
  loss = ft.history['loss']
  val_loss = ft.history['val_loss']
  epochs = range(len(accuracy))
  plt.plot(epochs, accuracy, 'bo', label='Training accuracy')
  plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
  plt.title('Training and validation accuracy')
  plt.legend()
  plt.figure()
  plt.plot(epochs, loss, 'bo', label='Training loss')
  plt.plot(epochs, val_loss, 'b', label='Validation loss')
  plt.title('Training and validation loss')
  plt.legend()
  plt.show()
  return


In [ ]:
## Results for small model with 1 layer
plot_results(small_1_layer_model, fit_s_1)

In [ ]:
## Results for medium model with 1 layer
plot_results(medium_1_layer_model, fit_m_1)

In [ ]:
## Results for large model with 1 layer
plot_results(large_1_layer_model, fit_l_1)

<font color='red'>**Task 1**: Do the same for models with 2 and 3 hidden layers </font>

<font color='red'>**Task 2**:
What are you conclusions from this experiment? which models present some overfit issue? which models to keep? </font>


# Batch normalization
-----

Here we will add a normalization batch step after each layer. An example using the following code.

In [ ]:
from keras.layers import BatchNormalization

model_w_norm = Sequential()
model_w_norm.add(Input(shape=(train_X[0].size,)))
model_w_norm.add(BatchNormalization())
model_w_norm.add(Dense(128, activation="relu"))
model_w_norm.add(BatchNormalization())
model_w_norm.add(Dense(64, activation="relu"))
model_w_norm.add(BatchNormalization())
model_w_norm.add(Dense(10, activation="softmax"))


In [ ]:
compiler(model_w_norm)
fit_w_norm = trainer(model_w_norm)

In [ ]:
plot_results(model_w_norm, fit_w_norm)

Now we can explore the batchnormalization on our 9 models

In [ ]:

# One layer models -----------------------------------------
## Small model

model_one_small = Sequential()
model_one_small.add(Input(shape=(train_X[0].size,)))
model_one_small.add(BatchNormalization())
model_one_small.add(Dense(16, activation="relu"))
model_one_small.add(BatchNormalization())
model_one_small.add(Dense(10, activation="softmax"))
compiler(model_one_small)
fit_one_small = trainer(model_one_small)

## Medium model
model_one_medium = Sequential()
model_one_medium.add(Input(shape=(train_X[0].size,)))
model_one_medium.add(BatchNormalization())
model_one_medium.add(Dense(64, activation="relu"))
model_one_medium.add(BatchNormalization())
model_one_medium.add(Dense(10, activation="softmax"))
compiler(model_one_medium)
fit_one_medium = trainer(model_one_medium)


## Large model
model_one_large = Sequential()
model_one_large.add(Input(shape=(train_X[0].size,)))
model_one_large.add(BatchNormalization())
model_one_large.add(Dense(256, activation="relu"))
model_one_large.add(BatchNormalization())
model_one_large.add(Dense(10, activation="softmax"))
compiler(model_one_large)
fit_one_large = trainer(model_one_large)

In [ ]:
plot_results(model_one_large, fit_one_large)

In [ ]:
# Two layer models -----------------------------------------
## Small model

model_two_small = Sequential()
model_two_small.add(Input(shape=(train_X[0].size,)))
model_two_small.add(BatchNormalization())
model_two_small.add(Dense(16, activation="relu"))
model_two_small.add(BatchNormalization())
model_two_small.add(Dense(8, activation="relu"))
model_two_small.add(BatchNormalization())
model_two_small.add(Dense(10, activation="softmax"))
compiler(model_two_small)
fit_two_small = trainer(model_two_small)

## Medium model
model_two_medium = Sequential()
model_two_medium.add(Input(shape=(train_X[0].size,)))
model_two_medium.add(BatchNormalization())
model_two_medium.add(Dense(64, activation="relu"))
model_two_medium.add(BatchNormalization())
model_two_medium.add(Dense(32, activation="relu"))
model_two_medium.add(BatchNormalization())
model_two_medium.add(Dense(10, activation="softmax"))
compiler(model_two_medium)
fit_two_medium = trainer(model_two_medium)


## Large model
model_two_large = Sequential()
model_two_large.add(Input(shape=(train_X[0].size,)))
model_two_large.add(BatchNormalization())
model_two_large.add(Dense(256, activation="relu"))
model_two_large.add(BatchNormalization())
model_two_large.add(Dense(128, activation="relu"))
model_two_large.add(BatchNormalization())
model_two_large.add(Dense(10, activation="softmax"))
compiler(model_two_large)
fit_two_large = trainer(model_two_large)

In [ ]:
plot_results(model_two_small, fit_two_small)

In [ ]:
# Three layer models -----------------------------------------
## Small model

model_three_small = Sequential()
model_three_small.add(Input(shape=(train_X[0].size,)))
model_three_small.add(BatchNormalization())
model_three_small.add(Dense(16, activation="relu"))
model_three_small.add(BatchNormalization())
model_three_small.add(Dense(8, activation="relu"))
model_three_small.add(BatchNormalization())
model_three_small.add(Dense(4, activation="relu"))
model_three_small.add(BatchNormalization())
model_three_small.add(Dense(10, activation="softmax"))
compiler(model_three_small)
fit_three_small = trainer(model_three_small)

## Medium model
model_three_medium = Sequential()
model_three_medium.add(Input(shape=(train_X[0].size,)))
model_three_medium.add(BatchNormalization())
model_three_medium.add(Dense(64, activation="relu"))
model_three_medium.add(BatchNormalization())
model_three_medium.add(Dense(32, activation="relu"))
model_three_medium.add(BatchNormalization())
model_three_medium.add(Dense(16, activation="relu"))
model_three_medium.add(BatchNormalization())
model_three_medium.add(Dense(10, activation="softmax"))
compiler(model_three_medium)
fit_three_medium = trainer(model_three_medium)


## Large model
model_three_large = Sequential()
model_three_large.add(Input(shape=(train_X[0].size,)))
model_three_large.add(BatchNormalization())
model_three_large.add(Dense(256, activation="relu"))
model_three_large.add(BatchNormalization())
model_three_large.add(Dense(128, activation="relu"))
model_three_large.add(BatchNormalization())
model_three_large.add(Dense(64, activation="relu"))
model_three_large.add(BatchNormalization())
model_three_large.add(Dense(10, activation="softmax"))
compiler(model_three_large)
fit_three_large = trainer(model_three_large)

In [ ]:
plot_results(model_three_small, fit_three_small)

# Reguralization

Reguralization is generally a good practice for overfitting issues. Here we explore $L_2$ reguralization.

In [ ]:
from keras.regularizers import l2

model_w_reg = Sequential()
model_w_reg.add(Input(shape=(train_X[0].size,)))
model_w_reg.add(BatchNormalization())
model_w_reg.add(Dense(256, activation="relu", kernel_regularizer=l2(0.001)))
model_w_reg.add(BatchNormalization())
model_w_reg.add(Dense(10, activation="softmax"))
compiler(model_w_reg)
fit_w_reg = trainer(model_w_reg)

In [ ]:
plot_results(model_w_reg, fit_w_reg)

In [ ]:
plot_results(model_one_large, fit_one_large)

**Question:** Has $L_2$ reguralization improved the performance?


# Dropout

Another possible option for addressing overfitting is the dropout.

In [ ]:
from keras.layers import Dropout

model_w_dropout = Sequential()
model_w_dropout.add(Input(shape=(train_X[0].size,)))
model_w_dropout.add(BatchNormalization())
model_w_dropout.add(Dropout(0.4))
model_w_dropout.add(Dense(256, activation="relu", kernel_regularizer=l2(0.001)))
model_w_dropout.add(BatchNormalization())
model_w_dropout.add(Dropout(0.4))
model_w_dropout.add(Dense(128, activation="relu", kernel_regularizer=l2(0.001)))
model_w_dropout.add(BatchNormalization())
model_w_dropout.add(Dropout(0.4))
model_w_dropout.add(Dense(64, activation="relu", kernel_regularizer=l2(0.001)))
model_w_dropout.add(BatchNormalization())
model_w_dropout.add(Dropout(0.4))
model_w_dropout.add(Dense(10, activation="softmax"))
compiler(model_w_dropout)
fit_w_dropout = trainer(model_w_dropout)

In [ ]:
plot_results(model_w_dropout, fit_w_dropout)

In [ ]:
plot_results(model_three_large, fit_three_large)

# Early stop

You can also adust the number of “epoch” by adding callback_early_stopping(patience = 5) to stop training if the loss has not improved after 5 epochs.

In [ ]:
from keras.callbacks import EarlyStopping
from keras import initializers
from keras.callbacks import EarlyStopping

model_w_callback = Sequential()
model_w_callback.add(Input(shape=(train_X[0].size,)))
model_w_callback.add(BatchNormalization())
model_w_callback.add(Dense(256, activation="relu", kernel_initializer=initializers.random_normal(stddev=0.01), kernel_regularizer=l2(0.001)))
model_w_callback.add(BatchNormalization())
#model_w_callback.add(Dropout(0.4))
model_w_callback.add(Dense(128, activation="relu", kernel_initializer=initializers.random_normal(stddev=0.01),  kernel_regularizer=l2(0.001)))
model_w_callback.add(BatchNormalization())
#model_w_callback.add(Dropout(0.4))
model_w_callback.add(Dense(64, activation="relu",kernel_initializer=initializers.random_normal(stddev=0.01), kernel_regularizer=l2(0.001)))
model_w_callback.add(BatchNormalization())
#model_w_callback.add(Dropout(0.4))
model_w_callback.add(Dense(10, activation="softmax"))
compiler(model_w_callback)
fit_w_callback = model_w_callback.fit(train_train_X, train_train_label, batch_size=128, epochs=10, verbose=1, validation_data=(train_valid_X, train_valid_label), callbacks=[EarlyStopping(patience=5)])


In [ ]:
plot_results(model_w_callback, fit_w_callback)


In [ ]:
keras.backend.clear_session()